<a href="https://colab.research.google.com/github/SebGalvaleira/mineria-datos-ii-parcial/blob/main/notebooks/e2/E2_03_silver.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# E2 · 03 · Silver: limpieza, conformado, calidad y enriquecimiento
Transforma Bronze en Silver: tipos limpios, unidades deducidas, features por métrica, join con organizaciones y recursos,
reglas de calidad duras (→ quarantine) y blandas (→ flags), conformado del maestro de organizaciones.

In [2]:
!pip install -q "pyspark>=4.0,<4.1"
import os, shutil, subprocess
os.environ.pop('JAVA_HOME', None)

from google.colab import drive
drive.mount('/content/drive')

from pathlib import Path
LAKE_DRIVE = Path('/content/drive/MyDrive/cpa_datalake')   # respaldo persistente
LAKE       = Path('/content/cpa_datalake')                  # copia de trabajo (disco local de Colab)

# Al iniciar la sesión: restaurar desde Drive si existe un respaldo
if not LAKE.exists() and LAKE_DRIVE.exists():
    shutil.copytree(LAKE_DRIVE, LAKE)
    print('Data Lake restaurado desde Drive')

RUTAS = {z: LAKE / z for z in ['landing', 'stream_input', 'bronze', 'silver',
                               'gold', 'quarantine', 'checkpoints', 'logs']}
for p in RUTAS.values():
    p.mkdir(parents=True, exist_ok=True)

def guardar_en_drive():
    """Sincroniza la copia local en Drive (llamar al final de cada notebook)."""
    LAKE_DRIVE.mkdir(parents=True, exist_ok=True)
    subprocess.run(['rsync', '-a', '--delete', f'{LAKE}/', f'{LAKE_DRIVE}/'], check=True)
    print('Data Lake sincronizado en Drive')

from pyspark.sql import SparkSession, functions as F, types as T
spark = (SparkSession.builder.appName('cloud-provider-analytics')
         .config('spark.sql.session.timeZone', 'UTC')
         .config('spark.sql.shuffle.partitions', '8')
         .config('spark.sql.sources.partitionOverwriteMode', 'dynamic')
         .getOrCreate())
print('Spark', spark.version)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Spark 4.0.5


In [3]:
#ller bronze
#ultimo_snapshot: si corre la ingesta batch otro dia, bronze tiene 2 fotos de los maestros (2 ingest_date). sin este filtro, el join duplicara cada evento
def ultimo_snapshot(df):
    """Los maestros son fotos diarias: nos quedamos con la más reciente."""
    ultima = df.agg(F.max('ingest_date')).first()[0]
    return df.filter(F.col('ingest_date') == ultima)

ev_bronze  = spark.read.parquet(str(RUTAS['bronze'] / 'usage_events'))
orgs_bronze = ultimo_snapshot(spark.read.parquet(str(RUTAS['bronze'] / 'customers_orgs')))
res_bronze  = ultimo_snapshot(spark.read.parquet(str(RUTAS['bronze'] / 'resources')))
print('eventos', ev_bronze.count(), '· orgs', orgs_bronze.count(), '· recursos', res_bronze.count())

eventos 43200 · orgs 80 · recursos 400


In [4]:
#silver del maestro de organizaciones
# NPS: guardamos el valor original (nps_score), una version valida solo si esta entre 0 y 10 (nps_0_10) y un flag
orgs_silver = (orgs_bronze
    .withColumn('industry',       F.lower(F.trim('industry')))
    .withColumn('plan_tier',      F.lower(F.trim('plan_tier')))
    .withColumn('flag_nps_fuera_escala', F.coalesce(~F.col('nps_score').between(0, 10), F.lit(False)))
    .withColumn('nps_0_10',       F.when(F.col('nps_score').between(0, 10), F.col('nps_score')))
    .select('org_id', 'org_name', 'industry', 'hq_region', 'plan_tier', 'is_enterprise',
            'signup_date', 'lifecycle_stage', 'nps_score', 'nps_0_10', 'flag_nps_fuera_escala',
            'source_file', 'ingest_ts'))

orgs_silver.write.mode('overwrite').parquet(str(RUTAS['silver'] / 'customers_orgs'))
orgs_silver.groupBy('flag_nps_fuera_escala').count().show()

+---------------------+-----+
|flag_nps_fuera_escala|count|
+---------------------+-----+
|                 true|   59|
|                false|   21|
+---------------------+-----+



In [5]:
#umbral de picos (se guarda como decision)
#el umbral no esta escrito a mano: se calcula con los datos y se guarda en un archivo
import json
p99 = ev_bronze.approxQuantile('cost_usd_increment', [0.99], 0.0)[0]
UMBRALES = {'p99_costo': p99, 'pico_costo': 5 * p99, 'costo_minimo_valido': -0.01}
with open(RUTAS['logs'] / 'umbrales.json', 'w') as f:
    json.dump(UMBRALES, f, indent=2)
UMBRALES

{'p99_costo': 16.7175, 'pico_costo': 83.5875, 'costo_minimo_valido': -0.01}

In [6]:
#la funcion de Silver
#try_cast: intenta convertir. si no puede devuelve nulo en vez de romper
#f.broadcast: las dimensiones son chicas (80 orgs, 400 recursos), asi que spark le manda un copia a cada maquina y evita un shuffle. es la forma eficiente de hacer un join stream
#f.coalesce(*[F.when(...)]): recorre las reglas en orden y se queda con el primer motivo que falle, si ninguna falla, queda nulo y el evento va a silver
UNIDAD_POR_METRICA = {'requests': 'count', 'cpu_hours': 'hours', 'storage_gb_hours': 'gb_hours'}
MAPA_UNIDAD = F.create_map(*[F.lit(x) for par in UNIDAD_POR_METRICA.items() for x in par])

ORGS_DIM = F.broadcast(orgs_silver.select('org_id', 'org_name', 'industry', 'plan_tier',
                                          'hq_region', 'is_enterprise'))
RES_DIM  = F.broadcast(res_bronze.select('resource_id', F.col('state').alias('resource_state'),
                                         F.col('created_at').alias('resource_created_at')))

def construir_silver(ev):
    # Regla "event_id único": dedupe (los nulos se separan para no fusionarlos entre sí)
    sin_id = ev.filter(F.col('event_id').isNull())
    con_id = ev.filter(F.col('event_id').isNotNull()).dropDuplicates(['event_id'])
    ev = con_id.unionByName(sin_id)

    ev = (ev
        # --- conformado ---
        .withColumn('value_num', F.expr('try_cast(value as double)'))
        .withColumn('unit_deducida', F.col('unit').isNull() & F.col('metric').isNotNull())
        .withColumn('unit', F.coalesce('unit', MAPA_UNIDAD[F.col('metric')]))
        # --- features ---
        .withColumn('requests',         F.when(F.col('metric') == 'requests',         F.col('value_num')))
        .withColumn('cpu_hours',        F.when(F.col('metric') == 'cpu_hours',        F.col('value_num')))
        .withColumn('storage_gb_hours', F.when(F.col('metric') == 'storage_gb_hours', F.col('value_num')))
        # genai_tokens y carbon_kg ya vienen; quedan nulos en v1 (no se inventan ceros)
        # --- reglas blandas: flags ---
        .withColumn('flag_costo_anomalo', F.col('cost_usd_increment') < UMBRALES['costo_minimo_valido'])
        .withColumn('flag_pico_costo',    F.col('cost_usd_increment') > UMBRALES['pico_costo'])
        # --- enriquecimiento (joins con dimensiones) ---
        .join(ORGS_DIM, 'org_id', 'left')
        .join(RES_DIM,  'resource_id', 'left'))

    # --- reglas duras: el primer motivo que falle manda el evento a quarantine ---
    REGLAS_DURAS = [
        ('event_id_nulo',           F.col('event_id').isNull()),
        ('value_no_numerico',       F.col('value').isNotNull() & F.col('value_num').isNull()),
        ('unit_nula_con_value',     F.col('value').isNotNull() & F.col('unit').isNull()),
        ('schema_version_invalida', F.col('schema_version').isNull() | ~F.col('schema_version').isin(1, 2)),
        ('org_desconocida',         F.col('org_name').isNull()),
    ]
    ev = ev.withColumn('motivo', F.coalesce(*[F.when(cond, F.lit(nombre)) for nombre, cond in REGLAS_DURAS]))

    silver     = ev.filter(F.col('motivo').isNull()).drop('motivo')
    quarantine = ev.filter(F.col('motivo').isNotNull()).withColumn('zona_origen', F.lit('silver'))
    return silver, quarantine

In [7]:
#recorre sobre todo bronze y escribe
#partitionBy('event_date', 'service'): tal como lo diseñamos en §7.2. Las consultas filtran por fecha y casi siempre por servicio, y hay solo 6 servicios (baja cardinalidad).
silver, quarantine = construir_silver(ev_bronze)
silver = silver.persist(); quarantine = quarantine.persist()

(silver.write.mode('overwrite').partitionBy('event_date', 'service')
       .parquet(str(RUTAS['silver'] / 'usage_events')))
if quarantine.count():
    quarantine.write.mode('overwrite').partitionBy('motivo').parquet(str(RUTAS['quarantine'] / 'silver_usage_events'))

n = {'bronze': ev_bronze.count(), 'silver': silver.count(), 'quarantine': quarantine.count()}
n['duplicados'] = n['bronze'] - n['silver'] - n['quarantine']
n['cuadra'] = n['bronze'] == n['silver'] + n['quarantine'] + n['duplicados']
n.update({
    'unit_deducida':      silver.filter('unit_deducida').count(),
    'value_nulo':         silver.filter(F.col('value').isNull()).count(),
    'flag_costo_anomalo': silver.filter('flag_costo_anomalo').count(),
    'flag_pico_costo':    silver.filter('flag_pico_costo').count(),
})

import datetime
with open(RUTAS['logs'] / 'silver_usage_events.jsonl', 'a') as f:
    f.write(json.dumps({'ts': datetime.datetime.now(datetime.timezone.utc).isoformat(), **n}) + '\n')
n

{'bronze': 43200,
 'silver': 43200,
 'quarantine': 0,
 'duplicados': 0,
 'cuadra': True,
 'unit_deducida': 2075,
 'value_nulo': 877,
 'flag_costo_anomalo': 211,
 'flag_pico_costo': 49}

In [8]:
#prueba de las reglas con casos controlados

# cada caso rompe una regla a proposito, lo esperado:
# quarantine: sin event_id, value no numeroco, org inexistente y version 9, cada uno con su motivo.
# silver con flags: el costo negativo (flag_costo_anomalo = True) y el pico (flag_pico_costo = True)
# el duplicado del evento real desaparece por la regla de unicidad: de original y duplicado queda uno solo

# estas muestras se guardan en quarantine/muestras_reglas, que es la evidencia de "quarantine con muestras" que pode la consigna
base = ev_bronze.limit(1).collect()[0].asDict()
casos = [
    {**base, 'event_id': None,                 '_caso': 'sin event_id'},
    {**base, 'event_id': 'evt_test_texto',     'value': 'abc',              '_caso': 'value no numérico'},
    {**base, 'event_id': 'evt_test_org',       'org_id': 'org_inexistente', '_caso': 'org que no existe'},
    {**base, 'event_id': 'evt_test_version',   'schema_version': 9,         '_caso': 'versión de esquema 9'},
    {**base, 'event_id': 'evt_test_negativo',  'cost_usd_increment': -50.0, '_caso': 'costo negativo'},
    {**base, 'event_id': 'evt_test_pico',      'cost_usd_increment': 500.0, '_caso': 'pico de costo'},
    {**base,                                                                '_caso': 'duplicado de un evento real'},
]
esquema_casos = T.StructType(ev_bronze.schema.fields + [T.StructField('_caso', T.StringType())])
df_casos = spark.createDataFrame(casos, schema=esquema_casos)

s_test, q_test = construir_silver(df_casos.unionByName(ev_bronze.limit(1).withColumn('_caso', F.lit('original'))))
print('→ QUARANTINE (reglas duras)')
q_test.select('_caso', 'event_id', 'motivo').show(truncate=False)
print('→ SILVER con flags (reglas blandas)')
s_test.select('_caso', 'event_id', 'cost_usd_increment', 'flag_costo_anomalo', 'flag_pico_costo').show(truncate=False)

q_test.write.mode('overwrite').parquet(str(RUTAS['quarantine'] / 'muestras_reglas'))   # muestras verificables

→ QUARANTINE (reglas duras)
+--------------------+----------------+-----------------------+
|_caso               |event_id        |motivo                 |
+--------------------+----------------+-----------------------+
|org que no existe   |evt_test_org    |org_desconocida        |
|value no numérico   |evt_test_texto  |value_no_numerico      |
|versión de esquema 9|evt_test_version|schema_version_invalida|
|sin event_id        |NULL            |event_id_nulo          |
+--------------------+----------------+-----------------------+

→ SILVER con flags (reglas blandas)
+---------------------------+-----------------+------------------+------------------+---------------+
|_caso                      |event_id         |cost_usd_increment|flag_costo_anomalo|flag_pico_costo|
+---------------------------+-----------------+------------------+------------------+---------------+
|duplicado de un evento real|evt_55195m07uwwj |9.4035            |false             |false          |
|costo negativo

In [9]:
#verificar silver
s = spark.read.parquet(str(RUTAS['silver'] / 'usage_events'))
print('filas:', s.count(), '· event_id distintos:', s.select('event_id').distinct().count())
s.select('event_id', 'event_date', 'service', 'industry', 'plan_tier', 'metric', 'value', 'value_num',
         'unit', 'unit_deducida', 'requests', 'cpu_hours', 'genai_tokens', 'carbon_kg',
         'flag_costo_anomalo', 'flag_pico_costo').show(5, truncate=False)

s.groupBy('schema_version').agg(F.count('*').alias('eventos'),
                                F.count('carbon_kg').alias('con_carbon'),
                                F.count('genai_tokens').alias('con_tokens')).show()
!find "{RUTAS['silver']}/usage_events" -maxdepth 1 -type d | wc -l

filas: 43200 · event_id distintos: 43200
+----------------+----------+-------+----------+---------+--------+-----+---------+-----+-------------+--------+---------+------------+---------+------------------+---------------+
|event_id        |event_date|service|industry  |plan_tier|metric  |value|value_num|unit |unit_deducida|requests|cpu_hours|genai_tokens|carbon_kg|flag_costo_anomalo|flag_pico_costo|
+----------------+----------+-------+----------+---------+--------+-----+---------+-----+-------------+--------+---------+------------+---------+------------------+---------------+
|evt_0dnib6oyxivc|2025-08-22|compute|fintech   |pro      |requests|117.0|117.0    |count|false        |117.0   |NULL     |NULL        |0.0234   |false             |false          |
|evt_1077scmo48tr|2025-08-22|compute|fintech   |pro      |requests|138.0|138.0    |count|false        |138.0   |NULL     |NULL        |0.0276   |false             |false          |
|evt_16aekvq4rtfw|2025-08-22|compute|education |standa

In [10]:
guardar_en_drive()

Data Lake sincronizado en Drive


## Cierre · Silver

**Qué se hizo**
- **Maestro conformado:** `customers_orgs` con categorías normalizadas y NPS tratado sin inventar reglas: valor original (`nps_score`), versión válida 0–10 (`nps_0_10`) y `flag_nps_fuera_escala` (59 de 80 orgs). Decisión abierta D2.
- **Eventos conformados:** `value` → `value_num` con `try_cast`; `unit` deducida de `metric` (2.075 eventos); esquema v1/v2 unificado (v1: 10.800 eventos sin `carbon_kg` ni `genai_tokens`; v2: 32.400 con carbon, 3.132 con tokens).
- **Features:** `requests`, `cpu_hours` y `storage_gb_hours` (separadas por métrica), más `genai_tokens` y `carbon_kg`.
- **Enriquecimiento:** join *broadcast* con organizaciones (industria, plan, región, enterprise) y recursos (estado, alta).
- **Partición:** `event_date` y `service` (60 días).

**Reglas de calidad**
| Tipo | Regla | Acción | Resultado con datos reales |
|---|---|---|---|
| Dura | `event_id` no nulo y único | Quarantine / dedupe | 0 nulos · 0 duplicados |
| Dura | `value` numérico si existe | Quarantine | 0 |
| Dura | `unit` no nula si existe `value` (tras deducirla) | Quarantine | 0 (2.075 deducidas) |
| Dura | `schema_version` ∈ {1, 2} | Quarantine | 0 |
| Dura | `org_id` existe en el maestro | Quarantine | 0 |
| Blanda | `cost_usd_increment >= -0.01` | `flag_costo_anomalo` | **211** |
| Blanda | `cost_usd_increment <= 5 × p99` (83,59 USD) | `flag_pico_costo` | **49** |

**Conciliación:** Bronze 43.200 = Silver 43.200 + Quarantine 0 + duplicados 0

**Prueba con casos controlados** (`quarantine/muestras_reglas`): se inyectaron 7 eventos que rompen cada regla. Los 4 de reglas duras fueron a quarantine con su motivo (`event_id_nulo`, `value_no_numerico`, `org_desconocida`, `schema_version_invalida`); el costo de −50 y el de 500 quedaron en Silver con su flag; el duplicado de un evento real se eliminó.

**Umbrales** (`logs/umbrales.json`): p99 del costo = 16,72 USD → pico = 5 × p99 = 83,59 USD. Se calculan con los datos, no a mano (decisión abierta D3: evaluar MAD).

**Nota:** la regla de la consigna (`>= -0.01`) tolera 5 costos negativos de centavos (entre −0,01 y 0), por eso hay 211 flags y no 216 negativos.